<img src='./img/up_bann.png' style="display: block; margin: 0 auto;">

# Challenge #2 @ 12th CASSINI  Hackathon: Space-enabled crisis response and civil resilience

This notebook takes the single-date workflow from [APEx 101](./apex_101.ipynb) and runs it over seven dates of interest. For the complete breakdown, check out our [first notebook](./apex_101.ipynb).


## Scaling our Algorithm; Impact around the Kakhovka Dam

The Kakhovka Dam, in the Kherson Oblast of Ukraine, [collapsed on 6 June 2023](https://www.bbc.com/news/world-europe-65818705). The exposed lakebed did not simply become permanent bare ground. Within months, tens of thousands of hectares underwent rapid natural succession into dense pioneer **willow** (Salix) and **poplar forest**, while the surrounding **agricultural fields lost** canal **irrigation**.

After [two years]((https://ukraine.un.org/en/295829-flooding-was-%E2%80%98just-beginning%E2%80%99-kakhovka-dam-disaster-two-years)), the disaster created a humanitarian and ecological crisis, threatening disease outbreaks and prompting the collapse of ecosystems. 

The [APEx Bare Soil Index (BSI) algorithm](https://algorithm-catalogue.apex.esa.int/apps/bsi) can help us monitor how the reservoir’s loss affected the surrounding fields.

By the end of this notebook, you will be able to...

- Filter the dates of interest through the CDSE STAC catalogue
- Store this dates and run the [APEx BSI algorithm](https://algorithm-catalogue.apex.esa.int/apps/bsi) on the [CDSE openEO](https://dataspace.copernicus.eu/analyse/openeo) federation
- Stack the results into one time series with xarray
- Visualise them and have an overview of the changes in an easy, understandable manner.

### **Prerequisites:**

#### 🛠️ Dependencies 
As we will be working with the APEx algorithms, it is important to install the following dependencies in your desired environment:

```text
- git
- git+https://github.com/ESA-APEx/apex_algorithms.git # The up-to-date version of apex_algorithms
- openeo
- geopandas
- xarray
```
You can make use of the `environment.yml` available in this repository to run all the notebooks!

#### 🛰️ openEO 

It is also good, to have an understanding of [openEO UDPs](https://open-eo.github.io/openeo-python-client/udp.html), as the APEx algorithms are based on them. You can find examples that explain how to use the `openeo` library in their [Python Client Examples](https://github.com/Open-EO/openeo-community-examples/tree/main/python).

#### 🔐 Github Authentication 

The `esa_apex_toolbox` library makes use of **Repository access** wich reads remote GitHub content, not local catalogue changes!<br>

If your environment defines `GITHUB_TOKEN`, the token cell below picks it up. Otherwise you will be asked for one, and you can press Enter to continue without i.

With the packages installed, we can start. 🚀 🛰️

### Import Required libraries

In [ ]:
%matplotlib inline

import io
import os
from getpass import getpass

import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
import openeo
import pandas as pd
import xarray as xr
from esa_apex_toolbox.algorithms import GithubAlgorithmRepository
from openeo import MultiResult
from pystac_client import Client
from shapely.geometry import box

### Define Costum Functions

The following function will allow us to interactively visualise our results.

In [ ]:
def update_plot(change):
    t_idx = change["new"]
    with out:
        out.clear_output(wait=True)
        
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(
            full_timeseries.isel(t=t_idx).values,
            cmap="RdBu_r",
            vmin=-1,
            vmax=1,
            origin="upper",
            aspect="equal",
        )
        cbar = fig.colorbar(im, ax=ax, shrink=0.8)
        cbar.set_label("BSI")
        
        t_val = str(full_timeseries["t"].values[t_idx])[:10]
        ax.set_title(f"Time Step {t_idx}: {t_val}")
        ax.axis("off")
        
        plt.tight_layout()
        plt.show()

> ##### Remember
> Always **treat a token as a password**: never print it, put it in a committed file, or include it in notebook outputs. Revoke and replace it immediately if exposed; replace it when it expires. Authentication is applied to GitHub API directory requests, not raw catalogue or process-definition downloads.


We start by bringing in our github credentials and set the token to access our APEx Catalog.


In [ ]:
github_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
if not github_token:
    github_token = getpass('GitHub token (optional; Enter for unauthenticated access): ') or None

## 1️⃣ Accessing our interest algorithm

We connect to the catalogue and ask for the algorithm by its id. [APEx 101](./apex_101.ipynb) showed that the id of the Bare Soil Index is `bsi`.

In [ ]:
repo = GithubAlgorithmRepository(
    owner='ESA-APEx',
    repo='apex_algorithms',
    folder='algorithm_catalog',
    branch='main',
    token=github_token,
)

bsi_algorithm = repo.get_algorithm('bsi')
bsi_algorithm

Printing the algorithm shows its title, id, and links, but we only need the `udp_link`.
<br>

The algorithm’s `udp_link` points to the openEO process we will run within the APEx Catalogue. We take its `href` (the web address), and use it in to build the job.

In [ ]:
bsi_link = bsi_algorithm.udp_link.href
bsi_link

## 2️⃣ Defining the area of interes: Kakhovka Dam

Now that we have already retrieved our algorithm, we need to define the time frame and the area where we will run it.

Kakhovka Dam is located at **46°46′40″N 33°22′13″E**. The bounding box below covers the dam and the fields around it. It is about 25 km wide and 24 km high, roughly 60,000 hectares.

Coordinates are written as west, south, east, north, in decimal degrees.

In [ ]:
geometry = (33.272095, 46.671585, 33.609924, 46.889762)  # west, south, east, north
shape = {'west': geometry[0], 'south': geometry[1], 'east': geometry[2], 'north': geometry[3]}

# to visualise our aoi
aoi = gpd.GeoSeries(box(*geometry), crs='EPSG:4326')
aoi.explore(color='red')

We search for Sentinel-2 images with 1.5% cloud cover or less, following these phases:

| Step / Phase | Target Date Window | Rationale & Expected Spectral Behavior |
| :--- | :--- | :--- |
| **Baseline (Pre-breach)** | May 2023 | Full reservoir pool (low BSI); surrounding winter wheat/rapeseed at high green biomass. |
| **Immediate Post-Drainage** | June 15 – June 25, 2023 | Lakebed drained; wet silt may keep BSI lower than bare soil alone would suggest |
| **Desiccation Peak** | July – August 2023 | Sediment dries into bare flats/sandbars. Represents the **maximum BSI peak** for the exposed reservoir bed. |
| **Pioneer Colonization** | September – October 2023 | Initial carpet of pioneer willow (*Salix*) and weed seedlings emerges; BSI begins dropping (NDVI rises). |
| **Year 1 Peak Biomass** | July 2024 | First full summer post-drainage; dense sapling canopy should bring BSI negative |
| **Year 2 Closed Canopy** | July 2025 | Young floodplain forest canopy closes over the former reservoir bed; surrounding un-irrigated farmland reveals chronic summer moisture stress. |
| **Multi-Year Benchmark** | July 2026 | Phenologically matched benchmark to distinguish permanent bottomland afforestation from surrounding rainfed/abandoned crop parcels. |

We search four periods: May to October 2023, and July of 2024, 2025 and 2026. From these we select seven dates.

- 5 months of 2023
- July month for 2024, 2025 and 2026

In [ ]:
dates_ranges = ["2023-05-05/2023-10-05",
              "2024-07-01/2024-07-31",
              "2025-07-01/2025-07-31",
              "2026-07-01/2026-07-31"]

max_cloud_cover = 1.5  # percentage


### Filtering interest dates

The [CDSE STAC Catalogue](https://browser.stac.dataspace.copernicus.eu/) provides information on the metadata of the whole Sentinel-2 L2A collection, not only for our area. The search is therefore a first filter. We still check the preview of each candidate date.<br>

The BSI algorithm uses Sentinel-2 imagery to identify patches of land that are bare soil; hence, we can filter Sentinel-2 imagery for clear images. We will look for images containing less than 1.5% cloud coverage.

In [ ]:
# we connect to Copernicus Data Space STAC catalog
stac_url = "https://catalogue.dataspace.copernicus.eu/stac"
client = Client.open(stac_url)

# filter metadata based on the cloud cover filter
search = [client.search(
    collections=["sentinel-2-l2a"],
    bbox=geometry,
    datetime= date_r,
    query={"eo:cloud_cover": {"lte": max_cloud_cover}},
    sortby=[{"field": "properties.datetime", "direction": "asc"}],
    limit=60,
) for date_r in dates_ranges ]

We will see that we get around 50 images that match the cloud percentage we are looking for over period of interest. We will choose the ones that allign with the date ranges we defined earlier:

In [ ]:
for date_range, search in zip(dates_ranges, search):
    print(date_range)
    for item in search.item_collection():
        print(f"  {item.datetime.date()}  {item.properties['eo:cloud_cover']:.1f}% cloud")
    print()

From each period we pick the clearest scene closest to the target window. We check the preview in the [CDSE browser](https://browser.dataspace.copernicus.eu/) before we commit to a date.

In [ ]:
dates_of_interest = [
    '2023-05-06',  # pre-breach baseline
    '2023-06-20',  # just after the breach
    '2023-08-24',  # drying bed
    '2023-09-13',  # early colonisation
    '2024-07-14',  # year 1
    '2025-07-27',  # year 2
    '2026-07-19',  # year 3
]

## 3️⃣ Accesing algorithm through CDSE openEO federation

Once we have defined the dates we are interested in, we will run the BSI algorithm workflow we defined in the [APEX 101 notebook](./apex_101.ipynb). This time we will batch the calculation of our 7 time steps through the CDSE openEO federation. Our job will be composed of:

1. Connect to the backend
2. Build the BSI process for each date
3. Collect the saved nodes into one job
4. Run the job
5. Match and stack the results
6. Visualise the series

The first step is connecting to our openEO CDSE federation backend.

In [ ]:
connection = openeo.connect("openeofed.dataspace.copernicus.eu").authenticate_oidc() # 1. connecting

Once connected, we will start our job definition.

For more on openEO processes and the CDSE federation, see the [CDSE openEO documentation](https://openeo.org/documentation/1.0/). [APEx 101](./apex_101.ipynb) explains user-defined processes.

We define our process over our interest dates, add a node for each time step and select the output format we want for the results.

In [ ]:
save_nodes = []

for date_str in dates_of_interest: # 2. build bsi process:
    cube = connection.datacube_from_process(
        "BSI",
        namespace=bsi_link,
        temporal_extent=[f"{date_str}T00:00:00Z", f"{date_str}T23:59:59Z"],
        spatial_extent=shape
    )
    
    # 3. save each calculation as a separate NetCDF (item)
    save_node = cube.save_result(
        format="NETCDF",
        options={"filename_prefix": f"bsi_{date_str}"}
    )
    save_nodes.append(save_node)


The following lines help us create our job and save multiple results (as each of our BSI calculations runs individually)

Then, we can visualise our defined openEO job before running it:

In [ ]:
# we create and launch only one batch job containing all 7 save nodes
job = connection.create_job(
    process_graph=MultiResult(save_nodes),  
    title="BSI_kakhovka_dam"
)
job

## 4️⃣ Running the Job

Each date runs as its own BSI calculation and returns one NetCDF file, so the job returns seven files.

With the `start_and_wait` function, we will be able to visualise the progress of our job in the CDSE openEO backend.

In [ ]:
job.start_and_wait() # 4. with this line we run our job and visualise the results 

The run takes several minutes, depending on the queue, and uses processing credits from your CDSE account. Your CDSE dashboard shows the cost of each job.

We can then retrieve our results from the items URL that openEO generates for us by calling the `get_results` function, followed by the `get_metadata` function. This will allow us to access our data cube with xarray and visualise our results.

In [ ]:
# 5. here we access our href to fetch and concatenate in an xarray our proccessed data
results = job.get_results()
stac_json = results.get_metadata()

hrefs = []
print("All available asset keys in STAC metadata:")
for asset_key, asset_val in stac_json.get("assets", {}).items():
    hrefs.append(asset_val.get('href'))
    print(f"- Retrieved: {asset_key}")

Next, we open each result file and stack the seven layers into one time series. All files cover the same area, so their `latitude` and `longitude` coordinates line up, and xarray can join them along the date dimension `t`.


In [ ]:
time_slices = [] # 6. we stack our results

for i, href in enumerate(hrefs):
    # add asset bytes directly using the authenticated openEO session
    response = connection.session.get(href)
    response.raise_for_status()
    
    # bring  into memory using the h5netcdf engine
    ds_mf = xr.open_dataset(io.BytesIO(response.content), engine="h5netcdf")
    
    # select the primary data variable (skipping coordinate metadata like 'crs')
    var_name = [v for v in ds_mf.data_vars if v != "crs"][0]
    layer_da = ds_mf[var_name]
    
    # drop any existing single-element time dimension to avoid dimension collisions (openEO results)
    for dim_name in ["t", "time"]:
        if dim_name in layer_da.dims:
            layer_da = layer_da.squeeze(dim_name, drop=True)
            
    # assign the explicit target timestamp
    target_date = dates_of_interest[i]
    layer_da = layer_da.expand_dims(t=[pd.to_datetime(target_date)])
    
    time_slices.append(layer_da)

full_timeseries = xr.concat(time_slices, dim="t")
full_timeseries.name = "bsi"

# sort chronologically 
full_timeseries = full_timeseries.sortby("t")


## 5️⃣ Visualising

The slider moves through the seven dates. Drag it to compare the layers.

In [ ]:
# display to avoid flickering
out = widgets.Output()

slider = widgets.IntSlider(
    min=0, 
    max=len(full_timeseries["t"]) - 1, 
    step=1, 
    value=0, 
    description="Time:"
)

# update function
slider.observe(update_plot, names="value")

# display the slider and output area together
display(slider, out)
update_plot({"new": 0})

### How to read the series

- **Colour**: red means high BSI (bare soil). Blue means low BSI (vegetation or water).
- **Compare the third date with the first**: if the drying bed is visible, it should look redder in August 2023 than in May 2023.
- **Follow the later dates**: does the former bed become bluer as vegetation establishes?
- **Test the hypotheses**: which expectations match the map, and which do not?

## 6️⃣ Takeaways



We started with one algorithm in the [**APEx catalogue**](https://algorithm-catalogue.apex.esa.int/) and built a monitoring workflow that runs on the [CDSE openEO federation](https://documentation.dataspace.copernicus.eu/APIs/openEO/federation/openeo_federation.html) around it. 

Our workflow can be easily reused by:

1. **Choosing the dates first**. A STAC search for clear scenes narrows the choice before any processing runs. The dates themselves are picked by hand from that search.
2. **Batch the work**. The **BSI** for seven dates.
3. **Read the change with its limits**. The slider shows how bare soil in the former reservoir bed changed over three years. It shows land-surface change, not its cause, and the cloud filter works on whole scenes.


To apply the same workflow to another place or event, we can easily change the area and the dates.

The same steps apply to every algorithm in the [**APEx catalogue**](https://algorithm-catalogue.apex.esa.int/) that runs in the CDSE openEO federation! Once you know the pattern, you can use it to act on what the satellites observe, and to recognise what is happening on the ground.


## Ready for the next application?

[APEx](https://apex.esa.int/) gives you a catalogue of ESA-validated Earth observation algorithms that run directly on Sentinel data.<br>

Choose an algorithm, define your area and dates, and the cloud platform returns a result you can map and analyse. @ the [12th CASSINI Hackathon](https://www.cassini.eu/hackathons/), you decide what to recognise and what to act on.

<img src='./img/low_bann.png' style="display: block; margin: 0 auto;">